In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

In [ ]:
# Import MNIST DataSet
from tensorflow.keras.datasets import mnist

# Load MNIST
(X_train, y_train), (X_test, y_test) = mnist.load_data()

print("Training images:", X_train.shape)
print("Testing images:", X_test.shape)

In [ ]:
# Normalize pixels from [0, 255] to [0, 1]
X_train = X_train.astype(np.float32) / 255.0
X_test = X_test.astype(np.float32) / 255.0

# Flatten 28x28 images into 784-dimensional vectors
X_train = X_train.reshape(X_train.shape[0], 784)
X_test = X_test.reshape(X_test.shape[0], 784)

print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)

In [ ]:
# Original Image
plt.imshow(X_train[0].reshape(28, 28), cmap="gray")
plt.title("Original Image")
plt.axis("off")
plt.show()

In [ ]:
def add_noise(X, noise_probability=0.3):
    noisy_X = X.copy()
    
    # Generate random values between 0 and 1
    random_values = np.random.rand(*X.shape)
    
    # Pixels where random value < probability become 0
    noisy_X[random_values < noise_probability] = 0
    
    return noisy_X

In [ ]:
noise_probability = 0.3

X_train_noisy = add_noise(X_train, noise_probability)
X_test_noisy = add_noise(X_test, noise_probability)

print("Original shape:", X_train.shape)
print("Noisy shape:", X_train_noisy.shape)

In [ ]:
# Original vs Noisy Figure
plt.figure(figsize=(8, 4))

plt.subplot(1, 2, 1)
plt.imshow(X_train[0].reshape(28, 28), cmap="gray")
plt.title("Original")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(X_train_noisy[0].reshape(28, 28), cmap="gray")
plt.title("Noisy")
plt.axis("off")

plt.show()

In [ ]:
def relu(x):
    return np.maximum(0, x)


def relu_derivative(x):
    return (x > 0).astype(float)


def sigmoid(x):
    return 1 / (1 + np.exp(-x))

In [ ]:
# Input -> Hidden
W1 = np.random.randn(784, 128) * 0.01
b1 = np.zeros((1, 128))

# Hidden -> Output
W2 = np.random.randn(128, 784) * 0.01
b2 = np.zeros((1, 784))

print("W1 shape:", W1.shape)
print("W2 shape:", W2.shape)

In [ ]:
def forward(X):
    # Encoder
    Z1 = X @ W1 + b1
    A1 = relu(Z1)
    
    # Decoder
    Z2 = A1 @ W2 + b2
    A2 = sigmoid(Z2)
    
    cache = (Z1, A1, Z2, A2)
    
    return A2, cache

In [ ]:
def mse_loss(Y, prediction):
    return np.mean((Y - prediction) ** 2)

In [ ]:
def backward(X, Y, cache):
    Z1, A1, Z2, A2 = cache
    m = X.shape[0]

    # MSE derivative
    dA2 = (2 / m) * (A2 - Y)
    
    # Sigmoid derivative
    dZ2 = dA2 * A2 * (1 - A2)
    
    dW2 = A1.T @ dZ2
    db2 = np.sum(dZ2, axis=0, keepdims=True)

    
    dA1 = dZ2 @ W2.T
    
    dZ1 = dA1 * relu_derivative(Z1)
    
    dW1 = X.T @ dZ1
    db1 = np.sum(dZ1, axis=0, keepdims=True)
    
    return dW1, db1, dW2, db2

In [ ]:
# Use a smaller dataset initially
X_train_small = X_train[:10000]
X_train_noisy_small = X_train_noisy[:10000]

print(X_train_small.shape)

In [ ]:
# Training
learning_rate = 0.1
epochs = 20
batch_size = 100

loss_history = []

for epoch in range(epochs):
    # Shuffle training data
    indices = np.random.permutation(len(X_train_small))
    
    X_train_small = X_train_small[indices]
    X_train_noisy_small = X_train_noisy_small[indices]
    
    total_loss = 0
    
    # Mini-batch training
    for start in range(0, len(X_train_small), batch_size):
        end = start + batch_size
        
        X_batch = X_train_noisy_small[start:end]
        Y_batch = X_train_small[start:end]
        
        # Forward propagation
        predictions, cache = forward(X_batch)
        
        # Calculate loss
        loss = mse_loss(Y_batch, predictions)
        total_loss += loss
        
        # Backpropagation
        dW1, db1, dW2, db2 = backward(X_batch, Y_batch, cache)
        
        # Gradient descent
        W1 -= learning_rate * dW1
        b1 -= learning_rate * db1
        
        W2 -= learning_rate * dW2
        b2 -= learning_rate * db2
    
    average_loss = total_loss / (len(X_train_small) / batch_size)
    
    loss_history.append(average_loss)
    
    print("Epoch:", epoch + 1, "Loss:", average_loss)

In [ ]:
# Plot Loss Curve
plt.plot(loss_history)

plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Autoencoder Training Loss")

plt.grid(True)
plt.show()

In [ ]:
# Testing
reconstructed, _ = forward(X_test_noisy[:10])

In [ ]:
plt.figure(figsize=(12, 8))

for i in range(5):
    
    # Original
    plt.subplot(3, 5, i + 1)
    plt.imshow(
        X_test[i].reshape(28, 28),
        cmap="gray"
    )
    plt.title("Original")
    plt.axis("off")
    
    # Noisy
    plt.subplot(3, 5, i + 6)
    plt.imshow(
        X_test_noisy[i].reshape(28, 28),
        cmap="gray"
    )
    plt.title("Noisy")
    plt.axis("off")
    
    # Reconstructed
    plt.subplot(3, 5, i + 11)
    plt.imshow(
        reconstructed[i].reshape(28, 28),
        cmap="gray"
    )
    plt.title("Reconstructed")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Reconstruction Error
test_loss = mse_loss(
    X_test[:10],
    reconstructed
)

print("Reconstruction error:", test_loss)